In [2]:
#  1. Install Required Library (only if not already installed)
!pip install transformers

#  2. Import Libraries
import pandas as pd
import re
from transformers import pipeline

#  3. Load Dataset and Take a Sample of 500 Tweets
df = pd.read_csv("tweets-data.csv")  # Make sure this file exists in your working directory
df_sample = df.sample(n=500, random_state=42).reset_index(drop=True)

#  4. Define Text Cleaning Function (Same as VADER practice)
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text)          # Remove URLs
    text = re.sub(r"@\w+", "", text)             # Remove mentions
    text = re.sub(r"#", "", text)                # Remove hashtag symbol
    text = re.sub(r"[^A-Za-z\s]", "", text)      # Remove special characters/numbers
    text = text.lower().strip()                  # Lowercase and strip whitespace
    return text

# Apply text cleaning
df_sample['cleaned_text'] = df_sample['Tweets'].apply(clean_text)

#  5. Load Transformer Pipeline
sentiment_classifier = pipeline("sentiment-analysis")

#  6. Define Sentiment Scoring Function with Token Limit Handling
def sentiment_scores_transformers(text):
    text = text.strip()
    if not text:
        return "NEUTRAL", 0.0

    # Truncate to 512 characters (to avoid token overflow)
    max_length = 512
    text = text[:max_length]

    result = sentiment_classifier(text)[0]
    return result['label'], result['score']

#  7. Apply Sentiment Function and Create New Columns
df_sample[['sentiment_label', 'sentiment_score']] = df_sample['cleaned_text'].apply(
    lambda x: pd.Series(sentiment_scores_transformers(x))
)

#  8. (Optional) Save the Result to a New CSV
df_sample.to_csv("tweets_with_transformer_sentiment.csv", index=False)

#  9. Display Sample Output
print(df_sample[['Tweets', 'cleaned_text', 'sentiment_label', 'sentiment_score']].head())

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


                                              Tweets  \
0  Le #DessinDePresse de Sanaga : ls sont morts c...   
1  #Russia #Wagner #RussiaCivilWar https://t.co/P...   
2  Exclusive content -https://t.co/oEiSIIB2Z1\n.\...   
3  Auch heute geht die politische Nachricht des T...   
4  @crazyclipsonly Same type that would take a ho...   

                                        cleaned_text sentiment_label  \
0  le dessindepresse de sanaga  ls sont morts com...        NEGATIVE   
1                       russia wagner russiacivilwar        NEGATIVE   
2  exclusive content \n\ncosplay japan titan tita...        NEGATIVE   
3  auch heute geht die politische nachricht des t...        NEGATIVE   
4  same type that would take a homemade playstati...        NEGATIVE   

   sentiment_score  
0         0.981537  
1         0.962062  
2         0.961531  
3         0.975570  
4         0.994473  
